In [1]:
from scipy.signal import decimate
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Dropout, Dense, GlobalAveragePooling2D, InputLayer,Flatten
import os
import random
import h5py

In [2]:
def get_dataset_name(file_name_with_dir):
    
    filename_without_dir = file_name_with_dir.split('/')[-1]
    print(filename_without_dir)
    temp = filename_without_dir.split('_')[:-1]
    print(temp)
    dataset_name = "_".join(temp)
    return dataset_name
filename_path="./data/Intra/train/rest_105923_1.h5"
with h5py. File (r"./data/Intra/train/rest_105923_1.h5" , 'r') as f :
    dataset_name = get_dataset_name(filename_path)
    print(dataset_name)
    matrix = f.get(dataset_name)[()]
    print(type(matrix ))
    print(matrix.shape)

rest_105923_1.h5
['rest', '105923']
rest_105923
<class 'numpy.ndarray'>
(248, 35624)


In [3]:
def z_score_normalization(data):
    mean = data.mean(axis=0)
    std = data.std(axis=0)
    return (data - mean) / std

In [4]:
def segment_data(data, label, window_size=500, stride=500):
    segments = []
    labels = []
    for start in range(0, data.shape[1] - window_size + 1, stride):
        end = start + window_size
        segment = data[:, start:end]
        segments.append(segment)
        labels.append(label)
    return segments, labels

In [5]:
import re

def infer_label_from_filename(filename, label_map):
    filename = filename.lower().replace('\\', '/')
    basename = os.path.basename(filename)
    
    for key in label_map:
        if key in basename:
            return label_map[key]
    
    raise ValueError(f"Could not infer label from filename: {filename}")

In [ ]:
def load_and_preprocess(filepath, label_map, num_chunks=50, downsample_factor=20):
    import h5py
    from scipy.signal import decimate

    task_label = infer_label_from_filename(filepath, label_map)

    with h5py.File(filepath, 'r') as f:
        datasetname = list(f.keys())[0]
        data = f.get(datasetname)[()]  # shape: (248, T)


    data = decimate(data, q=downsample_factor, axis=1)

    total_length = data.shape[1]
    chunk_length = total_length // num_chunks

    segments = []
    for i in range(num_chunks):
        start = i * chunk_length
        end = start + chunk_length
        if end > total_length:
            break

        window = data[:, start:end]

        # z-score normalization per channel
        mean = window.mean(axis=1, keepdims=True)
        std = window.std(axis=1, keepdims=True)
        window = (window - mean) / (std + 1e-8)

        segments.append(window[..., np.newaxis])  # shape: (248, chunk_len, 1)

    labels = [task_label] * len(segments)
    return segments, labels


In [7]:
batch_size = 32

# Set filepaths and label map
data_dir='./data/Intra/train'
filepaths = [
    os.path.normpath(os.path.join(data_dir, fname))
    for fname in os.listdir(data_dir)
    if fname.endswith('.h5')
]
print(filepaths)
label_map = {
    'rest': 0,
    'math': 1,
    'story': 1,
    'story_math': 1,         
    'working_memory': 2,
    'memory': 2,
    'motor': 3
}

['data\\Intra\\train\\rest_105923_1.h5', 'data\\Intra\\train\\rest_105923_2.h5', 'data\\Intra\\train\\rest_105923_3.h5', 'data\\Intra\\train\\rest_105923_4.h5', 'data\\Intra\\train\\rest_105923_5.h5', 'data\\Intra\\train\\rest_105923_6.h5', 'data\\Intra\\train\\rest_105923_7.h5', 'data\\Intra\\train\\rest_105923_8.h5', 'data\\Intra\\train\\task_motor_105923_1.h5', 'data\\Intra\\train\\task_motor_105923_2.h5', 'data\\Intra\\train\\task_motor_105923_3.h5', 'data\\Intra\\train\\task_motor_105923_4.h5', 'data\\Intra\\train\\task_motor_105923_5.h5', 'data\\Intra\\train\\task_motor_105923_6.h5', 'data\\Intra\\train\\task_motor_105923_7.h5', 'data\\Intra\\train\\task_motor_105923_8.h5', 'data\\Intra\\train\\task_story_math_105923_1.h5', 'data\\Intra\\train\\task_story_math_105923_2.h5', 'data\\Intra\\train\\task_story_math_105923_3.h5', 'data\\Intra\\train\\task_story_math_105923_4.h5', 'data\\Intra\\train\\task_story_math_105923_5.h5', 'data\\Intra\\train\\task_story_math_105923_6.h5', 'data

In [ ]:
def data_generator(filepaths, label_map, batch_size, seq_len=10, num_chunks=50):
    from collections import Counter
    while True:
        random.shuffle(filepaths)
        all_segments, all_labels = [], []

        for filepath in filepaths:
            segments, labels = load_and_preprocess(filepath, label_map, num_chunks=num_chunks)
            all_segments.extend(segments)
            all_labels.extend(labels)


        usable_len = len(all_segments) - (len(all_segments) % seq_len)
        all_segments = all_segments[:usable_len]
        all_labels = all_labels[:usable_len]


        X = np.array(all_segments).reshape(-1, seq_len, all_segments[0].shape[0], all_segments[0].shape[1], 1)

        y = []
        for i in range(0, usable_len, seq_len):
            chunk_labels = all_labels[i:i+seq_len]
            majority = Counter(chunk_labels).most_common(1)[0][0]
            y.append(majority)
        y = np.array(y)


        idx = np.random.permutation(len(y))
        X, y = X[idx], y[idx]

        for i in range(0, len(X), batch_size):
            yield X[i:i + batch_size], y[i:i + batch_size]


In [9]:
segments, _ = load_and_preprocess(filepaths[0], label_map)
print(segments[0].shape)

(248, 35, 1)


In [10]:
def count_total_segments(filepaths, label_map, seq_len, num_chunks=50):
    total_sequences = 0
    for filepath in filepaths:
        _, labels = load_and_preprocess(filepath, label_map, num_chunks=num_chunks)
        num_chunks_in_file = len(labels)
        sequences = num_chunks_in_file // seq_len
        total_sequences += sequences
    return total_sequences


In [11]:
from sklearn.model_selection import train_test_split
train_files, val_files = train_test_split(filepaths, test_size=0.2, random_state=42)

In [12]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense, LSTM, TimeDistributed, BatchNormalization, Reshape
from keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

def build_cnn_lstm_model(input_shape=segments[0].shape, num_classes=4, l2_lambda=0.01):
    model = Sequential()

    
    model.add(TimeDistributed(Conv2D(32, (3, 3), activation='relu', padding='same'), input_shape=input_shape))
    model.add(TimeDistributed(MaxPooling2D(pool_size=(2, 2))))
    model.add(TimeDistributed(BatchNormalization()))
    model.add(TimeDistributed(Dropout(0.3)))

    model.add(TimeDistributed(Conv2D(64, (3, 3), activation='relu', padding='same')))
    model.add(TimeDistributed(MaxPooling2D(pool_size=(2, 2))))
    model.add(TimeDistributed(BatchNormalization()))
    model.add(TimeDistributed(Dropout(0.3)))

 
    model.add(TimeDistributed(Flatten()))


    model.add(LSTM(64, return_sequences=False))  # summarize sequence
    model.add(Dropout(0.5))


    model.add(Dense(64, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))


    model.compile(
        loss='sparse_categorical_crossentropy',
        optimizer=Adam(learning_rate=0.001),
        metrics=['accuracy']
    )

    return model


In [19]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam




batch_size = 16
seq_len = 10
num_chunks = 50
downsample_factor = 20
num_classes = 4

train_total_segments = count_total_segments(train_files, label_map, seq_len, num_chunks)
train_steps_per_epoch = train_total_segments // batch_size
train_gen = data_generator(train_files, label_map, batch_size, seq_len, num_chunks)

val_total_segments = count_total_segments(val_files, label_map, seq_len, num_chunks)
val_steps_per_epoch = val_total_segments // batch_size
val_gen = data_generator(val_files, label_map, batch_size, seq_len, num_chunks)


sample_segment, _ = load_and_preprocess(train_files[0], label_map, num_chunks)
time_per_chunk = sample_segment[0].shape[1]
input_shape = (seq_len, 248, time_per_chunk, 1)


model = build_cnn_lstm_model(input_shape=input_shape, num_classes=num_classes)


early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)


model.fit(
    train_gen,
    steps_per_epoch=train_steps_per_epoch,
    epochs=15,
    validation_data=val_gen,
    validation_steps=val_steps_per_epoch,
    callbacks=[early_stop]
)


Epoch 1/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 11s 984ms/step - accuracy: 0.1994 - loss: 1.3900 - val_accuracy: 0.1250 - val_loss: 1.4249
Epoch 2/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 2s/step - accuracy: 0.2980 - loss: 1.3676 - val_accuracy: 0.0000e+00 - val_loss: 1.4164
Epoch 3/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 2s/step - accuracy: 0.2664 - loss: 1.3882 - val_accuracy: 0.0000e+00 - val_loss: 1.4541
Epoch 4/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - accuracy: 0.3518 - loss: 1.3184 - val_accuracy: 0.0000e+00 - val_loss: 1.6297
Epoch 5/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step - accuracy: 0.2026 - loss: 1.5084 - val_accuracy: 0.0000e+00 - val_loss: 1.4661


In [17]:
def load_test_data(filepaths, label_map, seq_len=10, num_chunks=50):
    from collections import Counter
    all_segments, all_labels = [], []

    for filepath in filepaths:
        segments, labels = load_and_preprocess(filepath, label_map, num_chunks=num_chunks)
        all_segments.extend(segments)
        all_labels.extend(labels)

    usable_len = len(all_segments) - (len(all_segments) % seq_len)
    all_segments = all_segments[:usable_len]
    all_labels = all_labels[:usable_len]

    X = np.array(all_segments).reshape(-1, seq_len, all_segments[0].shape[0], all_segments[0].shape[1], 1)

    y = []
    for i in range(0, usable_len, seq_len):
        label_seq = all_labels[i:i+seq_len]
        majority = Counter(label_seq).most_common(1)[0][0]
        y.append(majority)

    y = np.array(y)
    return X, y

In [18]:
import glob

# Collect test files
test_folder = './data/Intra/test'
test_filepaths = glob.glob(os.path.join(test_folder, '*.h5'))
test_filepaths = [os.path.normpath(p) for p in test_filepaths]

# Load and preprocess test data
X_test, y_test = load_test_data(test_filepaths, label_map)
# Direct evaluation
loss, accuracy = model.evaluate(X_test, y_test, verbose=1)
print(f"Test Accuracy: {accuracy:.4f}")

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - accuracy: 0.2708 - loss: 1.3849 
Test Accuracy: 0.2500
